In [ ]:
# Install the Crunch CLI
%pip install --upgrade crunch-cli

# Setup your local environment
!crunch setup-notebook datacrunch-2 --token aaaabbbbccccddddeeeeffff

# Your model

## Setup

In [ ]:
# Imports
import os

import joblib  # == 1.3.2
import pandas as pd  # == 2.1.0
from sklearn.linear_model import LinearRegression


In [ ]:
import crunch

# Load the Crunch Toolings
crunch_tools = crunch.load_notebook()

## Strategy Implementation

### Utilities

Function used in both `train()` and `infer()`.

In [ ]:
def get_model_path(
    model_directory_path: str,
):
    return os.path.join(
        model_directory_path,
        f"model.joblib"
    )

get_model_path("resources")

In [ ]:
def get_feature_columns(
    X: pd.DataFrame,
):
    return [
        column
        for column in X.columns
        if column.startswith("Feature_")
    ]


### The `train()` Function

In this function, you build and train your model for making inferences on the test data. Your model must be stored in the `model_directory_path`.

This function will be called in a frequency that is defined by your `train frequency` parameter that you will define when deploying your model on the Crunch platform.

In [ ]:
def train(
    X_train: pd.DataFrame,
    y_train: pd.DataFrame,
    model_directory_path: str,
) -> None:
    feature_columns = get_feature_columns(X_train)

    model = LinearRegression()
    model.fit(X_train[feature_columns], y_train["target"])

    model_path = get_model_path(model_directory_path)
    joblib.dump(model, model_path)


### The `infer()` Function

In the inference function, your trained model (if any) is loaded and used to make predictions on test data.

This function will be called on every `moon` of the `Out-Of-Sample`.

In [ ]:
def infer(
    X_test: pd.DataFrame,
    model_directory_path: str,
) -> pd.DataFrame:
    prediction = X_test[["id", "moon"]].copy()

    model_path = get_model_path(model_directory_path)
    model = joblib.load(model_path)

    feature_columns = get_feature_columns(X_test)
    prediction["prediction"] = model.predict(X_test[feature_columns])
    prediction["prediction"] = prediction["prediction"].clip(-1, 1)

    return prediction


## Local testing

To make sure your `train()` and `infer()` function are working properly, you can call the `crunch.test()` function that will reproduce the cloud environment locally. <br />
Even if it is not perfect, it should give you a quick idea if your model is working properly.

In [ ]:
crunch_tools.test(
    # Uncomment to disable the forced first train
    # force_first_train=False,
    force_first_train=True,

    # Uncomment to set the training frequency
    # train_frequency=2,  # train every 2 moons
    train_frequency=0,

    # Uncomment to disable the determinism check
    # no_determinism_check=True,
)

## Results

Once the local tester is done, you can preview the result stored in `prediction/prediction.parquet`.

### Local scoring

You can call the function that the system uses to estimate your score locally.

A [Pearson correlation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html) will be computed against the **targets**.

**Note**:
- If all predictions are constant, the correlation will be undefined. In this case, the score will be set to `0`.
- Predictions must be between `-1` and `1`.